In [ ]:
# Cell 1: Colab Setup

# Import the HTML display module
from IPython.display import display, HTML

print("Setup complete.")


In [ ]:
# Cell 1: ControlNet Python Backend Setup & Function
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from PIL import Image
from io import BytesIO
import base64
from IPython.display import display, Javascript
from google.colab import output

# 1. Install required libraries
print("Installing Python dependencies...")
# Use the -q (quiet) flag to keep the output clean
!pip install diffusers transformers accelerate controlnet_aux xformers torch --quiet
!pip install --upgrade pillow # Ensure PIL/Pillow is up to date

# 2. Setup ControlNet and SD model
try:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        raise RuntimeError("GPU not found. Please ensure the runtime is set to GPU.")

    # Initialize Canny Preprocessor
    from controlnet_aux import CannyDetector
    processor = CannyDetector()

    # Load the Canny ControlNet
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-canny",
        torch_dtype=torch.float16 # Use half-precision for memory
    )

    # Load the base SD pipeline
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        torch_dtype=torch.float16
    ).to(device)

    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

    # CRITICAL MEMORY OPTIMIZATION FOR COLAB:
    pipe.enable_xformers_memory_efficient_attention()
    pipe.enable_model_cpu_offload()

    print(f"\n✅ ControlNet model loaded successfully on: {device} with memory optimizations.")

except RuntimeError as e:
    print(f"\n❌ Error: {e}")
    print("Generation will be disabled. FIX: Go to Runtime > Change runtime type > Select GPU.")
    # Define mock functions to prevent errors in JavaScript, but alert the user
    processor = lambda x: Image.new('RGB', (512, 512), color = 'red')
    pipe = lambda *args, **kwargs: type('MockPipe', (object,), {'images': [Image.new('RGB', (512, 512), color = 'red')]})()

except Exception as e:
    print(f"\n❌ Failed to load ControlNet models due to an unexpected error: {e}")
    processor = lambda x: Image.new('RGB', (512, 512), color = 'red')
    pipe = lambda *args, **kwargs: type('MockPipe', (object,), {'images': [Image.new('RGB', (512, 512), color = 'red')]})()


# 3. Define the Python function callable from JavaScript
def run_controlnet_generation(b64_image_data, prompt):
    # This function receives Base64 image data and a prompt from JavaScript
    try:
        # Decode the Base64 image
        image_bytes = base64.b64decode(b64_image_data.split(',')[1])
        init_image = Image.open(BytesIO(image_bytes)).convert("RGB")

        # Resize image for ControlNet (Standard 512x512)
        init_image = init_image.resize((512, 512))

        # Run the Canny pre-processor
        control_image = processor(init_image, detect_resolution=512, image_resolution=512)

        # Define generation parameters
        negative_prompt = "poor quality, bad design, blurry, low resolution, ugly, artifacts, multiple heads, multiple bodies, low detail, tiling, bad geometry"
        # Use a fixed seed for reproducible results (optional)
        generator = torch.Generator(device="cuda" if torch.cuda.is_available() else "cpu").manual_seed(42)

        # Run the ControlNet diffusion pipeline
        output_image = pipe(
            prompt,
            control_image,
            negative_prompt=negative_prompt,
            num_inference_steps=25,
            guidance_scale=7.5,
            generator=generator
        ).images[0]

        # Convert output image to base64 for JavaScript
        buffered = BytesIO()
        output_image.save(buffered, format="PNG")
        b64_output = base64.b64encode(buffered.getvalue()).decode("utf-8")

        # Send result back to JavaScript by executing a JavaScript function
        js_code = f"window.receiveControlNetOutput('data:image/png;base64,{b64_output}', 'success', 'Image generated successfully using ControlNet!');"
        display(Javascript(js_code))

    except Exception as e:
        error_msg = f"ControlNet generation failed: {e}. Check Colab console for details."
        print(f"PYTHON ERROR: {error_msg}")
        js_code = f"window.receiveControlNetOutput('', 'error', '{error_msg}');"
        display(Javascript(js_code))


# Register the Python function to be callable from the JavaScript kernel
output.register_callback('runControlNet', run_controlnet_generation)


In [ ]:
from IPython.display import display, HTML

html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Modernize AI: Transforming Tradition with AI</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
    <style>
        body {
            font-family: 'Inter', sans-serif;
        }
        /* Custom colors */
        .bg-eco-primary { background-color: #2E7D32; }
        .bg-eco-dark { background-color: #1B5E20; }
        .bg-eco-light { background-color: #A5D6A7; }
        .bg-eco-secondary { background-color: #C8E6C9; }
        .text-eco-primary { color: #2E7D32; }
        .text-eco-dark { color: #1B5E20; }
        .hover\:bg-eco-dark:hover { background-color: #1B5E20; }
        .focus\:border-eco-primary:focus { border-color: #2E7D32; }
        .focus\:ring-eco-primary\/50:focus { --tw-ring-color: rgba(46, 125, 50, 0.5); }

        /* Animation for chart bars */
        .chart-bar {
            transition: width 0.5s ease-in-out;
            animation: grow 0.8s ease-out forwards;
        }
        @keyframes grow {
            from { width: 0%; }
        }

        /* Hide step sections by default */
        .step-section { display: none; }
        .step-section.active { display: block; }

        /* Colab specific fix to make app visible within the output frame */
        .container {
            max-width: 100%;
            margin: 0;
            padding: 0;
        }
    </style>
</head>
<body class="bg-gray-50 text-gray-800">

    <div class="container mx-auto p-4 md:p-8">

        <header class="text-center mb-10">
            <h1 class="text-4xl md:text-5xl font-bold text-eco-dark mb-2">Modernize AI: Transforming Tradition with AI</h1>
        </header>

        <div id="loading-indicator" class="fixed inset-0 bg-black bg-opacity-50 z-50 flex items-center justify-center hidden">
            <div class="bg-white p-6 rounded-lg shadow-xl flex items-center space-x-4">
                <svg class="animate-spin h-8 w-8 text-eco-primary" xmlns="http://www.w3.org/2000/svg" fill="none" viewBox="0 0 24 24">
                    <circle class="opacity-25" cx="12" cy="12" r="10" stroke="currentColor" stroke-width="4"></circle>
                    <path class="opacity-75" fill="currentColor" d="M4 12a8 8 0 018-8V0C5.373 0 0 5.373 0 12h4zm2 5.291A7.962 7.962 0 014 12H0c0 3.042 1.135 5.824 3 7.938l3-2.647z"></path>
                </svg>
                <span id="loading-message" class="text-lg font-medium text-gray-700">Loading...</span>
            </div>
        </div>

        <div id="message-container" class="fixed top-5 right-5 z-50"></div>

        <div class="space-y-12">
            <section id="step1" class="step-section active">
                <h2 class="text-3xl font-semibold mb-6 text-gray-800 border-b pb-2">Step 1: Select Room</h2>
                <div class="grid grid-cols-2 md:grid-cols-4 gap-4">
                    <button onclick="selectRoom('Bedroom')" class="p-6 bg-eco-light/50 border-2 border-eco-light hover:bg-eco-primary hover:text-white transition rounded-xl shadow-md text-lg font-medium text-eco-dark">Bedroom</button>
                    <button onclick="selectRoom('Bathroom')" class="p-6 bg-eco-light/50 border-2 border-eco-light hover:bg-eco-primary hover:text-white transition rounded-xl shadow-md text-lg font-medium text-eco-dark">Bathroom</button>
                    <button onclick="selectRoom('Kitchen')" class="p-6 bg-eco-light/50 border-2 border-eco-light hover:bg-eco-primary hover:text-white transition rounded-xl shadow-md text-lg font-medium text-eco-dark">Kitchen</button>
                    <button onclick="selectRoom('Living Room')" class="p-6 bg-eco-light/50 border-2 border-eco-light hover:bg-eco-primary hover:text-white transition rounded-xl shadow-md text-lg font-medium text-eco-dark">Living Room</button>
                </div>
            </section>

            <section id="step2" class="step-section">
                <h2 class="text-3xl font-semibold mb-2 text-gray-800 border-b pb-2">Step 2: Select Material Category</h2>
                <p class="text-lg text-gray-600 mb-6">Current Room: <span id="selected-room" class="font-bold text-eco-dark"></span></p>
                <div class="grid grid-cols-2 sm:grid-cols-3 lg:grid-cols-5 gap-4">
                    <button onclick="selectCategory('Flooring')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Flooring</button>
                    <button onclick="selectCategory('Roofing')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Roofing</button>
                    <button onclick="selectCategory('Paints')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Paints</button>
                    <button onclick="selectCategory('Interior Doors and Trims')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Interior Doors and Trims</button>
                    <button onclick="selectCategory('Exterior Finishes')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Exterior Finishes</button>
                    <button onclick="selectCategory('Wall Structure')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Wall Structure</button>
                    <button onclick="selectCategory('Foundation')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Foundation</button>
                    <button onclick="selectCategory('Insulation')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Insulation</button>
                    <button onclick="selectCategory('Ceiling Finishes')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Ceiling Finishes</button>
                    <button onclick="selectCategory('Fixtures/Hardware')" class="p-4 bg-gray-100 hover:bg-eco-secondary transition rounded-xl text-md shadow-sm">Fixtures/Hardware</button>
                </div>
                <div class="mt-8 text-center">
                    <button onclick="goBack()" class="py-3 px-8 bg-gray-300 hover:bg-gray-400 text-gray-800 text-lg font-bold rounded-xl transition duration-200 shadow-lg">
                        ← Back
                    </button>
                </div>
            </section>

            <section id="step3" class="step-section">
                <h2 class="text-3xl font-semibold mb-2 text-gray-800 border-b pb-2">Step 3: Define Parameters</h2>
                <p class="text-lg text-gray-600 mb-6">Current Category: <span id="selected-category" class="font-bold text-eco-dark"></span></p>

                <div class="space-y-4 max-w-xl mx-auto">
                    <div>
                        <label for="budget-input" class="block text-sm font-medium text-gray-700">Target Budget (Rs per sq unit)</label>
                        <input type="number" id="budget-input" placeholder="e.g., 500" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm p-3 border focus:border-eco-primary focus:ring focus:ring-eco-primary/50">
                    </div>
                    <div>
                        <label for="climate-input" class="block text-sm font-medium text-gray-700">Climate Suitability (e.g., Hot & Humid, Cold & Dry, Temperate)</label>
                        <input type="text" id="climate-input" placeholder="e.g., Hot and Humid Climate" class="mt-1 block w-full rounded-md border-gray-300 shadow-sm p-3 border focus:border-eco-primary focus:ring focus:ring-eco-primary/50">
                    </div>
                    <button onclick="searchMaterials()" class="w-full py-3 mt-4 bg-eco-primary hover:bg-eco-dark text-white text-lg font-bold rounded-xl transition duration-200 shadow-lg">
                        Find Top 5 Eco-Materials
                    </button>
                </div>
                <div class="mt-8 text-center">
                    <button onclick="goBack()" class="py-3 px-8 bg-gray-300 hover:bg-gray-400 text-gray-800 text-lg font-bold rounded-xl transition duration-200 shadow-lg">
                        ← Back
                    </button>
                </div>

            </section>

            <section id="step4" class="step-section">
                <h2 class="text-3xl font-semibold mb-4 text-gray-800 border-b pb-2">Step 4: Top 5 Eco-Friendly Materials</h2>
                <p class="text-lg text-gray-600 mb-6">Select one material to proceed with visualization and comparison.</p>
                <div class="overflow-x-auto shadow-xl rounded-xl border border-gray-200">
                    <table class="min-w-full divide-y divide-gray-200">
                        <thead class="bg-eco-dark">
                            <tr>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Material Name</th>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Cost (Rs/unit)</th>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Durability</th>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Eco_score</th>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Certifications</th>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Climate Suitability</th>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Aesthetic</th>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Indoor Temp</th>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Supplier Links</th>
                                <th class="p-3 text-left text-xs font-semibold text-white uppercase tracking-wider">Action</th>
                            </tr>
                        </thead>
                        <tbody id="materials-table-body" class="bg-white divide-y divide-gray-100">
                            <tr><td colspan="10" class="text-center py-4 text-gray-500">No results yet. Please complete Step 3.</td></tr>
                        </tbody>
                    </table>
                </div>
                <div class="mt-8 text-center">
                    <button onclick="goBack()" class="py-3 px-8 bg-gray-300 hover:bg-gray-400 text-gray-800 text-lg font-bold rounded-xl transition duration-200 shadow-lg">
                        ← Back
                    </button>
                </div>

            </section>

            <section id="step5" class="step-section">
                <h2 class="text-3xl font-semibold mb-4 text-gray-800 border-b pb-2">Step 5: Visualize & Compare</h2>
                <p class="text-lg text-gray-600 mb-6">Selected Material: <span id="comparison-material-name" class="font-bold text-eco-dark"></span></p>
                <div class="grid lg:grid-cols-2 gap-8">
                    <div class="flex flex-col">
                        <h3 class="text-2xl font-semibold mb-3 text-gray-800">Visualizing the Material</h3>
                        <div id="material-image-container" class="min-h-[350px] bg-gray-200 rounded-xl shadow-inner flex items-center justify-center text-gray-500">
                            Image will be generated here.
                        </div>
                    </div>

                    <div>
                        <h3 class="text-2xl font-semibold mb-3 text-gray-800">Eco vs. Non-eco Comparison (Gemini API)</h3>
                        <div id="comparison-results">
                            Comparison table and chart will be rendered here.
                        </div>
                    </div>
                </div>
                <div class="mt-8 text-center">
                    <button onclick="showStep('step6')" class="py-3 px-8 bg-eco-dark hover:bg-black text-white text-lg font-bold rounded-xl transition duration-200 shadow-lg">
                        Continue to Building Transformation
                    </button>
                </div>
                <div class="mt-8 text-center">
                    <button onclick="goBack()" class="py-3 px-8 bg-gray-300 hover:bg-gray-400 text-gray-800 text-lg font-bold rounded-xl transition duration-200 shadow-lg">
                        ← Back
                    </button>
                </div>

            </section>

            <section id="step6" class="step-section">
                   <h2 class="text-3xl font-semibold mb-4 text-gray-800 border-b pb-2">Step 6: Building Eco-Transformation (Local ControlNet)</h2>
                   <p class="text-lg text-gray-600 mb-6">Upload an image of a building.</p>
                   <div class="max-w-4xl mx-auto space-y-6">
                        <div class="flex flex-col sm:flex-row gap-4 items-center justify-center p-6 border-2 border-dashed border-gray-300 rounded-xl bg-gray-50">
                            <input type="file" id="building-image-upload" accept="image/*" class="w-full sm:w-auto p-2 border border-gray-300 rounded-lg bg-white shadow-sm">
                            <button onclick="transformBuilding()" class="w-full sm:w-auto py-2 px-6 bg-eco-primary hover:bg-eco-dark text-white font-bold rounded-xl shadow-md transition duration-200">
                                Transform to Eco Building
                            </button>
                        </div>

                        <div class="grid lg:grid-cols-2 gap-6">
                            <div class="flex flex-col">
                                <h3 class="text-lg font-medium text-gray-600 mb-2">Original Image Preview</h3>
                                <div id="original-image-preview" class="min-h-[200px] bg-gray-200 rounded-xl shadow-inner flex items-center justify-center text-gray-500">
                                    Upload image here.
                                </div>
                            </div>
                            <div class="flex flex-col">
                                <h3 class="text-lg font-medium text-gray-600 mb-2">Eco-Friendly Design Output</h3>
                                <div id="building-image-output" class="min-h-[200px] bg-gray-200 rounded-xl shadow-inner flex items-center justify-center text-gray-500">
                                    Transformed image will appear here.
                                </div>
                            </div>
                        </div>
                   </div>
                   <div class="mt-8 text-center">
                        <button onclick="goBack()" class="py-3 px-8 bg-gray-300 hover:bg-gray-400 text-gray-800 text-lg font-bold rounded-xl transition duration-200 shadow-lg">
                            ← Back
                        </button>
                    </div>

            </section>

        </div>

    <script>
        // --- App State and Configuration ---
        const appState = {
            category: null,
            room: null,
            selectedMaterial: null,
        };

        // API configuration
        // 🚨 IMPORTANT: REPLACE THIS WITH YOUR GEMINI API KEY (for Steps 3 & 5 data)
        const apiKey = "";
        const API_TEXT_MODEL = "gemini-2.5-flash";

        // HUGGING FACE CONFIGURATION (FOR STEP 5 - MATERIAL TEXTURE IMAGE)
        // 🚨 IMPORTANT: REPLACE THIS WITH YOUR HUGGING FACE API KEY
        const HF_API_KEY = "";
        const HF_TEXT2IMG_MODEL = "stabilityai/stable-diffusion-xl-base-1.0";


        // --- Colab Kernel Communication for ControlNet (Step 6) ---

        // Global function to receive the image result from the Python backend (Cell 1)
        window.receiveControlNetOutput = function(base64ImageResult, type, message) {
            setLoading(false);
            const outputContainer = document.getElementById("building-image-output");

            if (type === 'success') {
                outputContainer.innerHTML = `<img src="${base64ImageResult}" class="rounded-xl shadow-lg w-full h-full object-cover"/>`;
                displayMessage('success', message);
            } else {
                outputContainer.innerHTML = `<span class="text-red-500 text-center p-4">⚠️ Error: ${message}</span>`;
                displayMessage('error', message);
            }
        }


        // --- UI Helper Functions ---

        function showStep(stepId) {
            document.querySelectorAll('.step-section').forEach(step => {
                step.classList.remove('active');
            });
            const activeStep = document.getElementById(stepId);
            if (activeStep) {
                activeStep.classList.add('active');
                window.scrollTo({ top: activeStep.offsetTop - 80, behavior: 'smooth' });
            }
        }

        function setLoading(isLoading, message = 'Loading...') {
            const indicator = document.getElementById('loading-indicator');
            const msgElement = document.getElementById('loading-message');
            if (isLoading) {
                msgElement.textContent = message;
                indicator.classList.remove('hidden');
            } else {
                indicator.classList.add('hidden');
            }
        }

        function displayMessage(type, message) {
            const container = document.getElementById('message-container');
            const bgColor = type === 'error' ? 'bg-red-500' : 'bg-eco-primary';
            const msgDiv = document.createElement('div');
            msgDiv.className = `${bgColor} text-white font-bold rounded-lg shadow-lg py-3 px-5 mb-2 transition-opacity duration-300`;
            msgDiv.textContent = message;
            container.appendChild(msgDiv);

            setTimeout(() => {
                msgDiv.style.opacity = '0';
                setTimeout(() => msgDiv.remove(), 300);
            }, 5000);
        }

        // --- Navigation ---

        function goBack() {
            const steps = ["step1", "step2", "step3", "step4", "step5", "step6"];
            const activeIndex = steps.findIndex(id => document.getElementById(id).classList.contains("active"));
            if (activeIndex > 0) {
                showStep(steps[activeIndex - 1]);
            }
        }

        // --- REAL API Fetcher with Exponential Backoff (For Gemini Text Calls) ---
        async function fetchWithBackoff(url, options, retries = 3, delay = 1000) {
            try {
                const response = await fetch(url, options);
                if (!response.ok) {
                    if (response.status >= 400 && response.status < 500) {
                        const errorBody = await response.json().catch(() => ({ error: { message: 'Non-JSON error response' } }));
                        console.error("API Client Error:", errorBody);
                        throw new Error(`API request failed: ${errorBody.error?.message || response.statusText || 'Client error'}`);
                    }
                    throw new Error(`API request failed with status ${response.status}`);
                }
                return await response.json();
            } catch (error) {
                if (retries > 0 && !(error.message.includes('Client error') || error.message.includes('401') || error.message.includes('403'))) {
                    console.warn(`Request failed, retrying in ${delay / 1000}s... (${retries} retries left)`);
                    await new Promise(resolve => setTimeout(resolve, delay));
                    return fetchWithBackoff(url, options, retries - 1, delay * 2);
                } else {
                    console.error("API request failed (final attempt or client error):", error);
                    throw error;
                }
            }
        }

        // --- Hugging Face Text-to-Image (Step 5 Material Texture) ---
        async function fetchAndDisplayHFImage(prompt, containerId) {
            const imgContainer = document.getElementById(containerId);
            imgContainer.innerHTML = 'Generating material texture image...';

            // Check for API Key
            if (HF_API_KEY === "YOUR_HUGGING_FACE_API_KEY_HERE" || !HF_API_KEY) {
                imgContainer.innerHTML = '⚠️ **HUGGING FACE KEY MISSING** Please replace the placeholder key in the code for Step 5 image generation.';
                displayMessage('error', 'Hugging Face API Key is missing or invalid.');
                return;
            }

            try {
                const response = await fetch(
                    `https://api-inference.huggingface.co/models/${HF_TEXT2IMG_MODEL}`,
                    {
                        method: "POST",
                        headers: {
                            "Authorization": `Bearer ${HF_API_KEY}`,
                            "Content-Type": "application/json",
                        },
                        // We use a small image size (512x512) for speed and economy
                        body: JSON.stringify({ inputs: prompt, parameters: { height: 512, width: 512 }, options: { wait_for_model: true } }),
                    }
                );

                if (!response.ok) {
                    const errorText = await response.text();
                    console.error("HF Text-to-Image Error:", errorText);
                    imgContainer.innerHTML = `⚠️ Error generating image: ${response.status}. Check console for details. **(Hint: Token/Model issue)**`;
                    throw new Error(`Image generation failed: ${response.statusText}`);
                }

                const blob = await response.blob();
                const imgUrl = URL.createObjectURL(blob);
                imgContainer.innerHTML = `<img src="${imgUrl}" alt="${prompt}" class="rounded-xl shadow-lg w-full h-full object-cover"/>`;
                displayMessage('success', 'Material texture generated successfully!');

            } catch (error) {
                 console.error("Failed to generate HF image:", error);
                 imgContainer.innerHTML = `⚠️ Generation failed. Check API Key/Model: ${error.message}`;
            }
        }

        // --- Step-by-Step Logic ---
        function selectRoom(roomName) {
            appState.room = roomName;
            document.getElementById('selected-room').textContent = roomName;
            showStep('step2');
        }

        function selectCategory(categoryName) {
            appState.category = categoryName;
            document.getElementById('selected-category').textContent = categoryName;
            showStep('step3');
        }

        async function searchMaterials() {
            setLoading(true, 'Searching for top eco-materials...');
            const budget = document.getElementById('budget-input').value || 'any';
            const climate = document.getElementById('climate-input').value || 'any';

            if (apiKey === "YOUR_GEMINI_API_KEY_HERE" || !apiKey) {
                displayMessage('error', 'GEMINI API Key is missing. Cannot fetch materials data. Using sample data.');
                setLoading(false);
                const mockMaterials = [
                        { Material_Name: "Bamboo Flooring (Sample)", Cost_Rs_per_sq_unit: "250", Durability: "High", Eco_score: 9, Certifications: "FSC, LEED", Climate_suitability: "Moderate", Aesthetic: "Modern", Indoor_temperature: "Cooling", Supplier_links: "#" },
                        { Material_Name: "Cork Flooring (Sample)", Cost_Rs_per_sq_unit: "350", Durability: "Medium", Eco_score: 10, Certifications: "GreenGuard", Climate_suitability: "All", Aesthetic: "Warm", Indoor_temperature: "Insulating", Supplier_links: "#" },
                    ];
                renderMaterialsTable(mockMaterials);
                showStep('step4');
                return;
            }


            const userQuery = `Find the top 5 eco-friendly building materials for ${appState.category} in a ${appState.room}. The target budget is around Rs ${budget} per sq unit, and the climate is ${climate}.`;

            const searchSchema = {
                type: "OBJECT",
                properties: {
                    "materials": {
                        "type": "ARRAY",
                        "items": {
                            "type": "OBJECT",
                            "properties": {
                                "Material_Name": { "type": "STRING" },
                                "Cost_Rs_per_sq_unit": { "type": "STRING" },
                                "Durability": { "type": "STRING" },
                                "Eco_score": { "type": "INTEGER" },
                                "Certifications": { "type": "STRING" },
                                "Climate_suitability": { "type": "STRING" },
                                "Aesthetic": { "type": "STRING" },
                                "Indoor_temperature": { "type": "STRING" },
                                "Supplier_links": { "type": "STRING" }
                            },
                            required: ["Material_Name", "Cost_Rs_per_sq_unit", "Durability", "Eco_score"]
                        }
                    }
                }
            };

            const payload = {
                contents: [{ parts: [{ text: userQuery }] }],
                systemInstruction: { parts: [{ text: "You are a sustainable architecture assistant. Provide results in the requested JSON format. Ensure Eco_score is an integer from 1 to 10." }] },
                generationConfig: { responseMimeType: "application/json", responseSchema: searchSchema }
            };

            const apiUrl = `https://generativelanguage.googleapis.com/v1beta/models/${API_TEXT_MODEL}:generateContent?key=${apiKey}`;

            try {
                const result = await fetchWithBackoff(apiUrl, { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify(payload) });
                const jsonText = result.candidates?.[0]?.content?.parts?.[0]?.text;
                if (!jsonText) throw new Error("API did not return valid JSON for materials.");
                const data = JSON.parse(jsonText);
                renderMaterialsTable(data.materials);
                showStep('step4');
            } catch (error) {
                console.error("Error searching materials (Failing over to mock data):", error);
                displayMessage('error', `Failed to fetch materials: ${error.message}. Displaying sample data.`);
                const mockMaterials = [
                        { Material_Name: "Bamboo Flooring (Sample)", Cost_Rs_per_sq_unit: "250", Durability: "High", Eco_score: 9, Certifications: "FSC, LEED", Climate_suitability: "Moderate", Aesthetic: "Modern", Indoor_temperature: "Cooling", Supplier_links: "#" },
                        { Material_Name: "Cork Flooring (Sample)", Cost_Rs_per_sq_unit: "350", Durability: "Medium", Eco_score: 10, Certifications: "GreenGuard", Climate_suitability: "All", Aesthetic: "Warm", Indoor_temperature: "Insulating", Supplier_links: "#" },
                    ];
                renderMaterialsTable(mockMaterials);
                showStep('step4');
            } finally {
                setLoading(false);
            }
        }

        function renderMaterialsTable(materials) {
            const tbody = document.getElementById('materials-table-body');
            tbody.innerHTML = '';
            if (!materials || materials.length === 0) {
                tbody.innerHTML = '<tr><td colspan="10" class="text-center py-4 text-gray-500">No materials found.</td></tr>';
                return;
            }

            materials.forEach(material => {
                const row = document.createElement('tr');
                row.className = 'border-b hover:bg-green-50 transition duration-150';

                // Create TradeIndia search link based on material name
                const supplierLink = `https://www.tradeindia.com/search.html?search=${encodeURIComponent(material.Material_Name)}`;

                row.innerHTML = `
                    <td class="p-3 text-sm font-medium text-eco-dark">${material.Material_Name}</td>
                    <td class="p-3 text-sm">${material.Cost_Rs_per_sq_unit}</td>
                    <td class="p-3 text-sm">${material.Durability}</td>
                    <td class="p-3 text-sm font-bold">${material.Eco_score}/10</td>
                    <td class="p-3 text-sm text-xs">${material.Certifications || 'N/A'}</td>
                    <td class="p-3 text-sm">${material.Climate_suitability}</td>
                    <td class="p-3 text-sm">${material.Aesthetic}</td>
                    <td class="p-3 text-sm">${material.Indoor_temperature}</td>
                    <td class="p-3 text-sm"><a href="${supplierLink}" target="_blank" class="text-blue-600 hover:underline text-xs truncate max-w-xs block">TradeIndia</a></td>
                    <td class="p-3 text-sm w-1">
                        <button onclick="selectMaterial('${material.Material_Name}')" class="bg-eco-primary hover:bg-eco-dark text-white text-xs font-semibold py-1 px-3 rounded-full transition duration-150 shadow-md">Select</button>
                    </td>
                `;
                tbody.appendChild(row);
            });
        }

        async function selectMaterial(materialName) {
            appState.selectedMaterial = materialName;
            document.getElementById('comparison-material-name').textContent = materialName;
            document.getElementById('material-image-container').innerHTML = '';
            document.getElementById('comparison-results').innerHTML = '';
            showStep('step5');

            // Generate the image and fetch the comparison data concurrently
            const prompt = `Close-up texture of seamless, clean, high-quality, ${materialName}, used as ${appState.category} in a modern, sustainable interior design. Photorealistic, 4k, architectural photography style.`;

            await Promise.all([
                fetchAndDisplayHFImage(prompt, 'material-image-container'),
                fetchComparison(materialName)
            ]).catch(error => {
                console.error("Error in Step 5 concurrent operations:", error);
            });
        }

        async function fetchComparison(materialName) {
            setLoading(true, `Fetching comparison for ${materialName}...`);
            const comparisonResults = document.getElementById('comparison-results');

            if (apiKey === "YOUR_GEMINI_API_KEY_HERE" || !apiKey) {
                // MOCK DATA FALLBACK (since key is missing)
                const mockData = {
                    "Non_Eco_Material_Name": "Traditional Cement Tile",
                    "Comparison_Metrics": [
                        // ADDED Eco_Display_Value and Non_Eco_Display_Value to mock data
                        { "Metric": "Cost (1=Cheapest)", "Eco_Value": 3, "Non_Eco_Value": 7, "Eco_Display_Value": "Rs 450/sq unit", "Non_Eco_Display_Value": "Rs 850/sq unit" },
                        { "Metric": "Durability (10=Highest)", "Eco_Value": 8, "Non_Eco_Value": 5, "Eco_Display_Value": "25 years", "Non_Eco_Display_Value": "15 years" },
                        { "Metric": "Eco_Score (10=Highest)", "Eco_Value": 9, "Non_Eco_Value": 2, "Eco_Display_Value": "9/10", "Non_Eco_Display_Value": "2/10" },
                        { "Metric": "Electricity Consumption (1=Lowest)", "Eco_Value": 2, "Non_Eco_Value": 7, "Eco_Display_Value": "1.5 kWh", "Non_Eco_Display_Value": "4.2 kWh" }
                    ]
                };
                renderComparison(materialName, mockData);
                setLoading(false);
                return;
            }

            // UPDATED USER QUERY to explicitly ask for both score and display value with units
            const userQuery = `Compare the eco-friendly material "${materialName}" (used as ${appState.category}) with a conventional alternative. Provide JSON data for the following metrics:
            1. Cost (1=Cheapest): Return the exact cost in Rs/sq unit in the Eco_Display_Value/Non_Eco_Display_Value fields, and a 1-10 score in the Eco_Value/Non_Eco_Value fields.
            2. Durability (10=Highest): Return the exact lifespan in years in the Eco_Display_Value/Non_Eco_Display_Value fields, and a 1-10 score in the Eco_Value/Non_Eco_Value fields.
            3. Eco_Score (10=Highest): Return the 1-10 score (e.g., 9/10) in the Eco_Display_Value/Non_Eco_Display_Value fields, and the raw score in the Eco_Value/Non_Eco_Value fields.
            4. Electricity_Consumption (1=Lowest): Return the exact consumption in kWh in the Eco_Display_Value/Non_Eco_Display_Value fields, and a 1-10 score in the Eco_Value/Non_Eco_Value fields.
            Ensure the eco-material values are better than the non-eco material values for all metrics as specified (e.g., higher score for Durability, lower score for Cost).`;

            // UPDATED comparisonSchema to include Display Value fields
            const comparisonSchema = {
                type: "OBJECT",
                properties: {
                    "Non_Eco_Material_Name": { "type": "STRING" },
                    "Comparison_Metrics": {
                        "type": "ARRAY",
                        "items": {
                            "type": "OBJECT",
                            "properties": {
                                "Metric": { "type": "STRING" },
                                // 1-10 score for the chart
                                "Eco_Value": { "type": "INTEGER" },
                                "Non_Eco_Value": { "type": "INTEGER" },
                                // Exact value with units for the table
                                "Eco_Display_Value": { "type": "STRING" },
                                "Non_Eco_Display_Value": { "type": "STRING" }
                            },
                            required: ["Metric", "Eco_Value", "Non_Eco_Value", "Eco_Display_Value", "Non_Eco_Display_Value"]
                        }
                    }
                }
            };

            const payload = {
                contents: [{ parts: [{ text: userQuery }] }],
                systemInstruction: { parts: [{ text: "You are a materials science expert. Provide a direct comparison in the requested JSON format." }] },
                generationConfig: { responseMimeType: "application/json", responseSchema: comparisonSchema }
            };

            const apiUrl = `https://generativelanguage.googleapis.com/v1beta/models/${API_TEXT_MODEL}:generateContent?key=${apiKey}`;

            try {
                const result = await fetchWithBackoff(apiUrl, { method: 'POST', headers: { 'Content-Type': 'application/json' }, body: JSON.stringify(payload) });
                const jsonText = result.candidates?.[0]?.content?.parts?.[0]?.text;
                if (!jsonText) { throw new Error("API did not return structured JSON."); }
                renderComparison(materialName, JSON.parse(jsonText));
            } catch (error) {
                console.error("Error fetching comparison (Failing over to mock data):", error);
                displayMessage('error', `Comparison data failed: ${error.message}. Displaying sample data.`);

                // MOCK DATA FALLBACK (must be repeated here for the catch block)
                const mockData = {
                    "Non_Eco_Material_Name": "Traditional Cement Tile",
                    "Comparison_Metrics": [
                        { "Metric": "Cost (1=Cheapest)", "Eco_Value": 3, "Non_Eco_Value": 7, "Eco_Display_Value": "Rs 450/sq unit", "Non_Eco_Display_Value": "Rs 850/sq unit" },
                        { "Metric": "Durability (10=Highest)", "Eco_Value": 8, "Non_Eco_Value": 5, "Eco_Display_Value": "25 years", "Non_Eco_Display_Value": "15 years" },
                        { "Metric": "Eco_Score (10=Highest)", "Eco_Value": 9, "Non_Eco_Value": 2, "Eco_Display_Value": "9/10", "Non_Eco_Display_Value": "2/10" },
                        { "Metric": "Electricity Consumption (1=Lowest)", "Eco_Value": 2, "Non_Eco_Value": 7, "Eco_Display_Value": "1.5 kWh", "Non_Eco_Display_Value": "4.2 kWh" }
                    ]
                };
                renderComparison(materialName, mockData);
                // END OF MOCK DATA FALLBACK

            } finally {
                setLoading(false);
            }
        }

        function renderComparison(ecoName, data) {
            const resultsContainer = document.getElementById('comparison-results');
            const nonEcoName = data.Non_Eco_Material_Name || "Conventional Material";
            const metrics = data.Comparison_Metrics || [];

            // Build comparison table (Uses exact display values)
            let tableHTML = `
                <table class="min-w-full border border-gray-200 rounded-lg overflow-hidden shadow-md mb-6">
                    <thead class="bg-eco-dark text-white">
                        <tr>
                            <th class="px-4 py-2 text-left text-sm font-semibold">Metric</th>
                            <th class="px-4 py-2 text-left text-sm font-semibold">${ecoName}</th>
                            <th class="px-4 py-2 text-left text-sm font-semibold">${nonEcoName}</th>
                        </tr>
                    </thead>
                    <tbody class="bg-white divide-y divide-gray-200">
            `;

            metrics.forEach(m => {
                // Use the new Display values for the table
                const ecoDisplay = m.Eco_Display_Value || `${m.Eco_Value}/10 (Score)`;
                const nonEcoDisplay = m.Non_Eco_Display_Value || `${m.Non_Eco_Value}/10 (Score)`;

                tableHTML += `
                    <tr>
                        <td class="px-4 py-2 text-gray-700 font-medium">${m.Metric}</td>
                        <td class="px-4 py-2 text-eco-dark font-semibold">${ecoDisplay}</td>
                        <td class="px-4 py-2 text-gray-800">${nonEcoDisplay}</td>
                    </tr>
                `;
            });

            tableHTML += `
                    </tbody>
                </table>
            `;

            // Bar chart visualization (still uses 1-10 score)
            let chartHTML = `
                <div class="space-y-3">
                    ${metrics.map(m => `
                        <div>
                            <p class="text-sm font-medium text-gray-700 mb-1">${m.Metric}</p>
                            <div class="relative h-4 bg-gray-200 rounded-full overflow-hidden">
                                <div class="absolute left-0 top-0 h-4 bg-eco-primary chart-bar rounded-full" style="width: ${m.Eco_Value * 10}%"></div>
                                <div class="absolute left-0 top-0 h-4 bg-red-400 chart-bar rounded-full opacity-40" style="width: ${m.Non_Eco_Value * 10}%"></div>
                            </div>
                            <div class="flex justify-between text-xs text-gray-500 mt-1">
                                <span>${ecoName}: ${m.Eco_Value}/10</span>
                                <span>${nonEcoName}: ${m.Non_Eco_Value}/10</span>
                            </div>
                        </div>
                    `).join('')}
                </div>
            `;

            resultsContainer.innerHTML = `
                ${tableHTML}
                <h4 class="text-lg font-semibold text-gray-800 mt-4 mb-2">Visual Comparison (1-10 Score)</h4>
                ${chartHTML}
            `;
        }

        // --- Step 6: Building Transformation Logic (Using Local ControlNet) ---
         async function transformBuilding() {
            const fileInput = document.getElementById("building-image-upload");
            const preview = document.getElementById("original-image-preview");

            if (!fileInput.files.length) {
                displayMessage("error", "Please upload a building image first.");
                return;
            }

            const file = fileInput.files[0];

            // Show the preview instantly
            preview.innerHTML = `<img src="${URL.createObjectURL(file)}" class="rounded-xl shadow-lg w-full h-full object-cover"/>`;

            setLoading(true, "Processing image");

            // 1. Convert the file to a Base64 Data URL
            const reader = new FileReader();
            reader.onload = function(event) {
                const base64Image = event.target.result; // Data URL (Base64)

                // 2. Define the transformation prompt
                const prompt = `A highly detailed photorealistic image of the same building, but with a modern, eco-friendly design, green roofs, solar panels, and climbing vines. Architectural photography style.`;

                // 3. CRITICAL: Invoke the Python function 'runControlNet' defined in Cell 1
                // This is the local execution part.
                google.colab.kernel.invokeFunction('runControlNet', [base64Image, prompt], {});
            };
            reader.readAsDataURL(file);
        }


        // Listener for the original image preview
        document.getElementById('building-image-upload').addEventListener('change', function() {
            const file = this.files[0];
            const previewContainer = document.getElementById('original-image-preview');
            const outputContainer = document.getElementById("building-image-output");
            if (file) {
                // Use URL.createObjectURL for instant local preview
                previewContainer.innerHTML = `<img src="${URL.createObjectURL(file)}" class="rounded-xl shadow-lg w-full h-full object-cover"/>`;
                outputContainer.innerHTML = 'Transformed image will appear here.'; // Reset output
            } else {
                previewContainer.innerHTML = 'Upload image here.';
            }
        });

    </script>
</body>
</html>
"""

# Display the HTML content in the Colab output
display(HTML(html_content))
